In [1]:
import pandas as pd
import numpy as np

# Load the parquet file
df = pd.read_parquet('../03_processed_datasets/chunk_00.parquet')

# Count occurrences of numHelped > 0 by phase
phase1_helped_count = len(df[(df['phase'] == 1) & (df['numHelped'] > 0)])
phase2_helped_count = len(df[(df['phase'] == 2) & (df['numHelped'] > 0)])

# Total rows by phase for comparison
phase1_total = len(df[df['phase'] == 1])
phase2_total = len(df[df['phase'] == 2])

# Print results
print(f"Phase 1: {phase1_helped_count}/{phase1_total} rows have numHelped > 0 ({phase1_helped_count/phase1_total*100:.2f}%)")
print(f"Phase 2: {phase2_helped_count}/{phase2_total} rows have numHelped > 0 ({phase2_helped_count/phase2_total*100:.2f}%)")

# Print a sample of each case if they exist
if phase1_helped_count > 0:
    sample_phase1 = df[(df['phase'] == 1) & (df['numHelped'] > 0)].iloc[0]
    print("\nSample row with phase=1 and numHelped>0:")
    print(sample_phase1[['event_id', 'phase', 'numHelped']])

if phase2_helped_count > 0:
    sample_phase2 = df[(df['phase'] == 2) & (df['numHelped'] > 0)].iloc[0]
    print("\nSample row with phase=2 and numHelped>0:")
    print(sample_phase2[['event_id', 'phase', 'numHelped']])
else:
    print("\nNo rows found with phase=2 and numHelped>0")

Phase 1: 62509/942546 rows have numHelped > 0 (6.63%)
Phase 2: 78809/942546 rows have numHelped > 0 (8.36%)

Sample row with phase=1 and numHelped>0:
event_id     1813
phase           1
numHelped       1
Name: 10, dtype: object

Sample row with phase=2 and numHelped>0:
event_id     1812
phase           2
numHelped       1
Name: 9, dtype: object


In [15]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Set display options to show all rows and columns with full width
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)  # Show full content of each cell
# Filtering Q nur wenn accepted answer received in Phase 2,

ddf = pd.read_parquet('../03_processed_datasets/chunk_01.parquet')
# ddf = pd.read_parquet('../03_processed_datasets/question_centered_model_allQ_7d.parquet')

# print(ddf.columns.tolist())

# note: agg_df["meaned_numHelped"] = agg_df.groupby("user_id")["numHelped"].transform(lambda x: x - x.mean())
# 1. Model: numHelped
# 2. Model: Question FE: question_id demeanen
# 3. Model: User FE: user_id demeanen
# -> should be correct for both cases right?

# Filter the dataframe to find rows where numHelped is not 0 and other specified columns are not 0
filtered_df = ddf[(ddf['numHelped'] != 0) &
                 (ddf['numQuestionsAskedAT'] != 0) &
                 (ddf['numHelpReceivedAT'] != 0) &
                 (ddf['numHelpProvidedAT'] != 0)
                ]

# Group by event_id and count the occurrences
event_count = filtered_df.groupby('event_id').size()

# Find event_ids that appear at least twice (both rows for the event_id meet criteria)
valid_events = event_count[event_count == 2].index.tolist()

if valid_events:
    sample_event_id = valid_events[0]
    print(f"Found event_id: {sample_event_id}")
    # Display the rows for this event_id
    print(filtered_df[filtered_df['event_id'] == sample_event_id])
else:
    print("No event_ids found that match all criteria in both rows.")


Found event_id: 44
   event_id  phase  user_id               timestamp            event  \
2        44      1        5 2008-08-08 15:48:58.527  Phase_One_Start   
3        44      2        5 2008-08-15 15:48:58.527  Phase_Two_Start   

   question_id         phase_one_start           phase_two_end  \
2        12406 2008-08-08 15:48:58.527 2008-08-22 15:48:58.527   
3        12406 2008-08-08 15:48:58.527 2008-08-22 15:48:58.527   

     event_history  is_history  has_answer  has_accepted_answer  \
2  Phase_One_Start           0           1                    1   
3  Phase_Two_Start           0           1                    1   

   time_to_first_answer_hours  time_to_accepted_answer_hours  \
2                    0.294694                       0.294694   
3                    0.294694                       0.294694   

   time_to_accept_vote_hours  numHelped  hasAnswer  numHelpProvidedEver  \
2                 -16.110952         10          1                    1   
3                 -1

No event_ids found that match all criteria in both rows.


In [14]:
"meaned_numHelped ~ C(phase) + C(has_accepted_answer) + C(phase):C(has_accepted_answer)"
# 1x ohne controls, 1x mit allen

'meaned_numHelped ~ C(phase) + C(has_accepted_answer) + C(phase):C(has_accepted_answer)'

In [3]:
# Model A) ohne filter
# Model B) Mit target erreichen: 3,5,8,10

target_value = 5
max_values = ddf.groupby('user_id')['numHelpProvidedAT'].max()
user_ids_with_target_max = max_values[max_values == target_value].index

# Option A
# df = ddf[ddf['user_id'].isin(user_ids_with_target_max)]

# Option B
df = ddf[ddf['numHelpProvidedAT']<=5]

# Create bins from 0 to 10
bin_edges = [-np.inf, 0, 1, 2, 3, 4, 5]
bin_labels = ['0', '1', '2', '3', '4', '5']

# Add bin column to dataframe
df['numHelpProvidedAT_bin'] = pd.cut(
   df['numHelpProvidedAT'],
   bins=bin_edges,
   labels=bin_labels,
   right=True
)

C:\Users\svenp\AppData\Local\Temp\ipykernel_5896\4047295048.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['numHelpProvidedAT_bin'] = pd.cut(


In [4]:
# Print descriptive statistics for mean value of "numHelped" for each bin and phase
print("Descriptive statistics for mean value of 'numHelped' by bin and phase:")
print(df.groupby(['phase'], observed=False)['numHelped'].mean())

print("Descriptive statistics for max value of 'numHelped' by bin and phase:")
print(df.groupby(['phase'], observed=False)['numHelped'].max())

# Additional statistics that might be useful
print("\nCount in each bin by phase:")
print(df.groupby(['phase'], observed=False)['numHelped'].count())

Descriptive statistics for mean value of 'numHelped' by bin and phase:
phase
1    0.073892
2    0.116449
Name: numHelped, dtype: float64
Descriptive statistics for max value of 'numHelped' by bin and phase:
phase
1    109
2     91
Name: numHelped, dtype: int32

Count in each bin by phase:
phase
1    747154
2    747154
Name: numHelped, dtype: int64


In [6]:
df.loc[df["has_accepted_answer"] == 0, "time_to_accepted_answer_hours"] = 0
df.loc[df["has_answer"] == 0, "time_to_first_answer_hours"] = 0

# Accepted Answer Model with numHelped

In [ ]:
# Remove rows where time_to_accepted_answer_hours is negative
rows_before = len(df)
df_cleaned = df[df['time_to_accepted_answer_hours'] >= 0]
rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after

print(f"Removed {rows_removed} rows where time_to_accepted_answer_hours was negative")
print(f"Original dataframe: {rows_before} rows")
print(f"Cleaned dataframe: {rows_after} rows")

# Remove rows where time_to_accepted_answer_hours is negative
rows_before = len(df_cleaned)
df_cleaned = df_cleaned[df_cleaned['time_to_accepted_answer_hours'] <= (7*24)]
rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after

print(f"Removed {rows_removed} rows where time_to_accepted_answer_hours was beyond 7 days")
print(f"Original dataframe: {rows_before} rows")
print(f"Cleaned dataframe: {rows_after} rows")

df_cleaned = df_cleaned.convert_dtypes()  # Convert columns to best possible dtypes
for col in df_cleaned.select_dtypes(include=["Int32", "Int64"]).columns:  # Find nullable integer columns
    df_cleaned[col] = df_cleaned[col].astype("int64")  # Convert to standard int64
for col in df_cleaned.select_dtypes(include=["Float32"]).columns:  # Handle nullable floats if needed
    df_cleaned[col] = df_cleaned[col].astype("float64")  # Convert to standard float64

In [32]:
df = df.convert_dtypes()  # Convert columns to best possible dtypes
for col in df.select_dtypes(include=["Int32", "Int64"]).columns:  # Find nullable integer columns
    df[col] = df[col].astype("int64")  # Convert to standard int64
for col in df.select_dtypes(include=["Float32"]).columns:  # Handle nullable floats if needed
    df[col] = df[col].astype("float64")  # Convert to standard float64

In [25]:
# Get counts for each combination
combination_counts = df.groupby(['phase', 'has_accepted_answer']).size().reset_index(name='count')

# Print the counts in a readable format
print("Counts for each combination of phase and has_accepted_answer:")
print("-----------------------------------------------------------")
for _, row in combination_counts.iterrows():
    phase = row['phase']
    has_answer = row['has_accepted_answer']
    count = row['count']
    print(f"Phase={phase}, has_accepted_answer={has_answer}: {count} observations")

# Print total count
total = ddf.shape[0]
print(f"\nTotal observations: {total}")

Counts for each combination of phase and has_accepted_answer:
-----------------------------------------------------------
Phase=1, has_accepted_answer=0: 3065811 observations
Phase=1, has_accepted_answer=1: 1709287 observations
Phase=2, has_accepted_answer=0: 3065811 observations
Phase=2, has_accepted_answer=1: 1709287 observations

Total observations: 9990242


In [29]:
# Calculate min, max, and median of meaned_numHelped
min_val = df["numHelpProvidedAT"].min()
max_val = df["numHelpProvidedAT"].max()
median_val = df["numHelpProvidedAT"].median()

# Print the results
print(f"numHelpProvidedAT statistics:")
print(f"Minimum: {min_val}")
print(f"Maximum: {max_val}")
print(f"Median: {median_val}")

numHelpProvidedAT statistics:
Minimum: 0
Maximum: 5
Median: 0.0


In [28]:
df[df["event_id"] == 14689210]

,event_id,phase,user_id,timestamp,event,question_id,phase_one_start,phase_two_end,event_history,is_history,...,numAcceptedAnswersReceived7D,numAcceptedVotesReceived7D,numQuestionsAsked3D,numHelpReceived3D,numHelpProvided3D,numAnswersReceived3D,numAcceptedAnswersReceived3D,numAcceptedVotesReceived3D,meaned_numHelped,numHelpProvidedAT_bin
4959168,14689210,1,5231999,2015-08-28 09:35:59.913,Phase_One_Start,32394723,2015-08-28 09:35:59.913,2015-09-11 09:35:59.913,Phase_One_Start,0,...,0,0,0,0,0,0,0,0,88.5,0
4959169,14689210,2,5231999,2015-09-11 09:35:59.913,Phase_Two_End,32394723,2015-08-28 09:35:59.913,2015-09-11 09:35:59.913,Phase_Two_End,0,...,0,0,0,0,0,0,0,0,-88.5,0


In [33]:
# Model
formula = ("meaned_numHelped ~ C(phase) + C(has_accepted_answer) + C(phase):C(has_accepted_answer)")

model = smf.ols(formula=formula, data=df).fit()

# Get clustered standard errors by User_ID
clustered_se = model.get_robustcov_results(
   cov_type='cluster',
   groups=df['user_id']
).cov_params()

# Print model summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       meaned_numHelped   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                 4.746e+04
Date:                Thu, 17 Apr 2025   Prob (F-statistic):               0.00
Time:                        00:43:27   Log-Likelihood:            -7.7726e+06
No. Observations:             9550196   AIC:                         1.555e+07
Df Residuals:                 9550192   BIC:                         1.555e+07
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

# Has Answer Model with numHelped

In [ ]:
# Remove rows where time_to_accepted_answer_hours is negative
rows_before = len(df)
df_cleaned = df[df['time_to_first_answer_hours'] > 0]
rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after

print(f"Removed {rows_removed} rows where time_to_first_answer_hours was negative")
print(f"Original dataframe: {rows_before} rows")
print(f"Cleaned dataframe: {rows_after} rows")

# Remove rows where time_to_accepted_answer_hours is negative
rows_before = len(df_cleaned)
df_cleaned = df_cleaned[df_cleaned['time_to_first_answer_hours'] <= (7*24)]
rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after

print(f"Removed {rows_removed} rows where time_to_first_answer_hours was beyond 7 days")
print(f"Original dataframe: {rows_before} rows")
print(f"Cleaned dataframe: {rows_after} rows")

df_cleaned = df_cleaned.convert_dtypes()  # Convert columns to best possible dtypes
for col in df_cleaned.select_dtypes(include=["Int32", "Int64"]).columns:  # Find nullable integer columns
    df_cleaned[col] = df_cleaned[col].astype("int64")  # Convert to standard int64
for col in df_cleaned.select_dtypes(include=["Float32"]).columns:  # Handle nullable floats if needed
    df_cleaned[col] = df_cleaned[col].astype("float64")  # Convert to standard float64

In [ ]:
# Define and fit the model
formula = ("meaned_numHelped ~ C(phase):C(has_answer) + "
           "C(phase):C(has_answer):np.log(time_to_first_answer_hours +1) + "
           "C(phase):C(has_answer):np.log(numQuestionsAskedAT + 1) + "
           "C(phase):C(has_answer):C(numHelpProvidedAT_bin) + "
           "C(year)")

model = smf.ols(formula=formula, data=df_cleaned).fit()

# Get clustered standard errors by User_ID
clustered_se = model.get_robustcov_results(
   cov_type='cluster',
   groups=df_cleaned['user_id']
).cov_params()

# Print model summary
print(model.summary())